# ROADGUARD — YOLOv8 License Plate Detection Training Pipeline
### AI-Powered Vehicle Document Verification System using ANPR

This notebook trains a custom YOLOv8s (Small) model for Indian License Plate localization on a **free Google Colab T4 GPU** in ~25 to 30 minutes.
- **Dataset**: 7,233 annotated images (Train: 5,116 | Val: 1,512 | Test: 605)
- **Model**: YOLOv8 Small (`yolov8s.pt`)
- **Target Metric**: mAP@0.5 > 98%, Inference Latency ~25ms

## 1. Verify Free GPU Acceleration (Tesla T4)
Ensure `Runtime` -> `Change runtime type` is set to **T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python matplotlib

## 3. Extract Dataset
Upload `unique_dataset.zip` directly to the Colab files panel on the left (or via Google Drive).

In [ ]:
import os
import glob
import zipfile
from pathlib import Path

# Auto-detect uploaded zip file (e.g. unique_dataset_combined.zip)
zip_candidates = glob.glob('*dataset*.zip') + glob.glob('unique*.zip') + glob.glob('*.zip')
if zip_candidates and not os.path.exists('unique dataset'):
    target_zip = zip_candidates[0]
    print(f"Unzipping {target_zip}...")
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Unzip complete!")

# Ensure standard folder hierarchy
if not os.path.exists('unique dataset') and os.path.exists('train'):
    os.makedirs('unique dataset', exist_ok=True)
    for folder in ['train', 'valid', 'test', 'data.yaml']:
        if os.path.exists(folder):
            os.rename(folder, os.path.join('unique dataset', folder))

DATASET_PATH = Path("unique dataset")
print(f"Dataset folder exists: {DATASET_PATH.exists()}")
print(f"Train images: {len(list((DATASET_PATH / 'train' / 'images').glob('*.*')))}")
print(f"Val images:   {len(list((DATASET_PATH / 'valid' / 'images').glob('*.*')))}")
print(f"Test images:  {len(list((DATASET_PATH / 'test' / 'images').glob('*.*')))}")

## 4. Train YOLOv8 Small (yolov8s) Detector (~25 to 30 minutes on Tesla T4 GPU)
Upgrading from `yolov8n` (3.2M params) to `yolov8s` (11.2M params) provides significantly richer spatial features, eliminating false positives on vehicle grilles.


In [ ]:
from ultralytics import YOLO
import torch

# 1. Load YOLOv8 Small architecture base weights (3.5x higher capacity than Nano)
model = YOLO("yolov8s.pt")

# 2. Train on 7,233 annotated vehicle images (Combined Dataset) with cosine LR schedule & early stopping
results = model.train(
    data="unique dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    patience=10,
    device=0 if torch.cuda.is_available() else "cpu",
    project="ROADGUARD_Plate_Detector",
    name="yolov8s_combined_detector",
    exist_ok=True
)


## 5. Evaluate on Unseen Test Dataset

In [ ]:
metrics = model.val(split="test")
print(f"=== ROADGUARD TEST EVALUATION RESULTS ===")
print(f"mAP@0.50:        {metrics.box.map50 * 100:.2f}%")
print(f"mAP@0.50:0.95:   {metrics.box.map * 100:.2f}%")
print(f"Precision (P):   {metrics.box.p[0] * 100:.2f}%")
print(f"Recall (R):      {metrics.box.r[0] * 100:.2f}%")

## 6. Display Training Convergence Graphs & Confusion Matrix

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path("ROADGUARD_Plate_Detector/yolov8s_combined_detector")
results_png = run_dir / "results.png"
if results_png.exists():
    print("Training Loss & Metric Curves:")
    display(Image(filename=str(results_png)))

conf_matrix = run_dir / "confusion_matrix.png"
if conf_matrix.exists():
    print("Confusion Matrix:")
    display(Image(filename=str(conf_matrix)))

## 7. Automatically Download `best.pt` & Results Package

In [ ]:
import os
import shutil
from google.colab import files

run_dir = "ROADGUARD_Plate_Detector/yolov8s_combined_detector"
if os.path.exists(run_dir):
    shutil.make_archive('trained_model_results', 'zip', run_dir)
    print("Downloading trained_model_results.zip (includes best.pt and all evaluation graphs)...")
    files.download('trained_model_results.zip')
    best_weights = os.path.join(run_dir, 'weights', 'best.pt')
    if os.path.exists(best_weights):
        files.download(best_weights)
else:
    print(f"Warning: {run_dir} not found. Check training logs above.")